![Platform+](images/banner.png)

# Lab 7: Getting Data In
## Stage → Format → COPY INTO → Query → Chart → Share

> **Persona track: Data Analyst** — same lab, your angle.
>
> Getting data in is only step one. An analyst's job doesn't end at a loaded table — it ends at an answer someone can act on. In this lab you'll follow one dataset through the everyday analyst loop: **land it → load it → model it into a view → analyze it → visualize it → share it.**

:point_right: **In this lesson, you'll bring two datasets into Snowflake: loyalty sign-ups and a supplier product catalog, both from stages. You'll define one reusable file format, load both files with COPY INTO — asking CoCo for help on the first and writing the second yourself — then chart your results and export them to share.**

The scenario: The Roastery loyalty programme has been running for months — sign-up data lives in an internal stage. Meanwhile, your coffee bean supplier has started dropping quarterly product catalog updates to a separate supplier stage. You need to get both into Snowflake so analysts can query them.

We'll cover:
1. Inspecting what's in a stage before loading
2. Defining a CSV file format and loading with COPY INTO — with CoCo's help
3. Reusing that format to load a second dataset — this time you write the load
4. What happens when a file has bad rows
5. Your angle — analyst queries, a quick chart, and exporting results to share

:books: **By the end you'll be able to:**
- Use `LIST` and `SELECT $1, $2, $3` to inspect a staged file before loading
- Define a FILE FORMAT and load data with COPY INTO — first with CoCo, then independently
- Diagnose a failed load and identify whether it's a source-data problem
- Create a view, summarize results, and visualize them as a bar chart and a line chart
- Replay query results with `RESULT_SCAN` and export them to CSV for stakeholders

**Time:** ~50 min · **Lab interface:** Workspace Notebook with Cortex Code (CoCo) as your tutor.
*Source data lives in the shared, read-only `PLATPLUS_DB`; everything you create lives in your own database, warehouse, and role — each named after your username (e.g. `USER_DB` / `USER_WH` / `USER_RL`).*

---
👉 **Connecting to your compute service:**
1. In the top bar, if no service is connected, click to connect and select your service.
2. Set your **role and warehouse** using the picker at the top-right — or just run the setup cell below, which sets them for you in SQL. (SQL cells run on the warehouse.)
3. Once the status shows **Connected** (green), run a cell with **Shift+Enter** (run + advance) or **Cmd/Ctrl+Enter** (run this cell only); the **▶** button works too.

> 📌 The service is a container that runs the notebook kernel and your **Python** cells. **SQL cells (and Snowpark pushdown) run on the query warehouse**, not the container. It stays running until it hits its idle timeout or you suspend it manually.
> 💡 **Best practice: Start a fresh chat for this lab.** Before you begin, open a fresh CoCo chat — click **New chat** (the **+** at the top of the panel). Each lab creates its own objects, so a fresh chat ensures CoCo isn't referencing tables or schemas from a prior lab that no longer exist. Shorter conversations also produce sharper answers and use fewer tokens.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
my_db = user + '_DB'
my_wh = user + '_WH'
my_role = user + '_RL'
print('Context ready. Signed in as ' + user)
print('Your workspace -> db: ' + my_db + ' | wh: ' + my_wh + ' | role: ' + my_role)


## Set your session context

Every query runs under a role (what you're allowed to do) and a warehouse (compute). The cells below create your personal warehouse and database if they don't exist yet, then switch into your own schema for this lab.

In [ ]:
USE ROLE {{my_role}};
CREATE WAREHOUSE IF NOT EXISTS {{my_wh}}
  WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE {{my_wh}};
CREATE DATABASE IF NOT EXISTS {{my_db}};
CREATE SCHEMA IF NOT EXISTS {{my_db}}.RAW;
USE SCHEMA {{my_db}}.RAW;


---
## 1 · Inspect Before You Load

Good loading practice: always look at the raw file before writing a single line of SQL. You need to know:
- What files are in the stage?
- How many columns are there?
- What are the delimiters and does the file have a header row?
- Are there any obvious issues in the data?

The stage `PLATPLUS_DB.ROASTERY_RAW.LOAD_STAGE` holds the loyalty sign-up data that's been accumulating since the programme launched. Let's see what's there.

In [ ]:
LIST @PLATPLUS_DB.ROASTERY_RAW.LOAD_STAGE;


> :mag_right: **Review the output.** Three files: `loyalty_signups.csv` (load this now), `loyalty_events.json` (used later for a JSON example), and `loyalty_signups_bad.csv` (a bad-row file used in another section). Note the signup file's size — you'll verify the row count after loading.

**Reading from a stage without loading.** The `SELECT` below doesn't move any data into Snowflake — it reads the raw file from the stage and returns each row as text. Nothing is written anywhere.

The `$1`, `$2`, `$3` notation is **positional**: Snowflake maps each delimited field in the file to a numbered column — `$1` is the first field, `$2` the second, and so on. Because no **FILE FORMAT** has been applied yet, every field comes back as a raw string. Adding `$4` shows the *boundary*: an empty `$4` means field 4 doesn't exist, confirming exactly 3 columns. Use this to answer the three questions you need: *How many columns are there? Does the file have a header row? What's the delimiter?*

> 💡 **What's a FILE FORMAT?** A FILE FORMAT is the small set of instructions that tells Snowflake how to interpret the shape of a file: its delimiter, whether it has a header row, how nulls are written. You'll create one in the next step.

In [ ]:
-- Positional preview: $1/$2/$3 = the three fields, $4 empty = no 4th column (boundary check)
SELECT $1, $2, $3, $4
FROM @PLATPLUS_DB.ROASTERY_RAW.LOAD_STAGE/loyalty_signups.csv
LIMIT 5;


**See the raw line.** Turn off delimiter parsing with `FIELD_DELIMITER='NONE'` so `$1` returns the whole line — now you can see the real delimiter and any quoting yourself.

In [ ]:
CREATE OR REPLACE FILE FORMAT {{my_db}}.RAW.NO_DELIM_FMT TYPE='CSV' FIELD_DELIMITER='NONE';
SELECT $1 AS raw_line
FROM @PLATPLUS_DB.ROASTERY_RAW.LOAD_STAGE/loyalty_signups.csv
  (FILE_FORMAT => '{{my_db}}.RAW.NO_DELIM_FMT')
LIMIT 5;


> :mag_right: **Your turn — read the file.**  
> Before building the FILE FORMAT you need to answer three questions about this file:
> 1. **How many columns are there?** (Look at which positional columns have values — and which is empty.)
> 2. **Does the file have a header row?** (Look at row 1: is it data or labels?)
> 3. **What's the delimiter?** (The filename is a clue — what does `.csv` stand for?)
>
> <details><summary>Check your read</summary>
>
> **Three columns** — `$1`, `$2`, and `$3` all have values; `$4` is empty in every row, marking the column boundary.  
> **Row 1 is the header** — it shows `CUSTOMER_ID`, `SIGNUP_DATE`, `CHANNEL` rather than actual values. You'll need to skip it with `SKIP_HEADER=1`.  
> **Comma-delimited CSV** — `.csv` stands for comma-separated values, so the delimiter is a comma. `CSV` is exactly the value you'll enter for `TYPE` in the FILE FORMAT cell coming up next. The date format is `YYYY-MM-DD` and channel is a short string (`app`, `web`, `in-store`).
>
> That's all the information you need to write the FILE FORMAT.
> </details>

---
## 2 · Define the Format, Then COPY INTO

In a mature data org, a platform engineer or data engineer owns the ingestion pipeline. You're writing this SQL today for two reasons: you're doing a one-off load for exploratory analysis, and once you've written it yourself you'll know exactly what to look for when a pipeline breaks and data goes missing. If you'd rather skip the SQL and load through the Snowsight UI, there's a note at the end of this section — both approaches produce the same result.

Snowflake uses **FILE FORMAT** objects to describe how a staged file is structured — delimiter, whether there's a header, how to handle nulls, etc. You define the format once and reuse it across any number of COPY INTO statements.

**Why this matters for analysts:** When a pipeline breaks or someone reports "the data looks wrong", the first place to check is the FILE FORMAT definition. Wrong delimiter, wrong null handling, or a missing `SKIP_HEADER` are the most common causes of malformed loads.

**Your turn — fill in the two blanks.** The cell below creates the FILE FORMAT object. Two values are left for you to supply, straight from what the peek told you:
> - `TYPE` — the *kind* of file you're loading. You confirmed it's a comma-separated-values file, so this is `CSV`.
> - `SKIP_HEADER` — how many rows at the top are labels rather than data. The peek showed a single header row, so this is `1`.
>
> The rest is pre-filled for you: `FIELD_DELIMITER=','` (fields split on commas), `TRIM_SPACE=TRUE` (strip stray spaces), `NULL_IF=('','NULL')` (treat blanks and the literal text `NULL` as null), and `EMPTY_FIELD_AS_NULL=TRUE`.

In [ ]:
-- Fill in the two blanks using what you learned from the peek above
CREATE OR REPLACE FILE FORMAT {{my_db}}.RAW.CSV_FMT
  TYPE            = ___   -- what kind of file is this? (hint: the .csv extension)
  SKIP_HEADER     = ___   -- how many header rows did the peek show?
  FIELD_DELIMITER = ','
  TRIM_SPACE      = TRUE
  NULL_IF         = ('', 'NULL')
  EMPTY_FIELD_AS_NULL = TRUE;


<details>
<summary>📖 Reference solution — stuck on the blanks?</summary>

    CREATE OR REPLACE FILE FORMAT {{my_db}}.RAW.CSV_FMT
      TYPE            = CSV    -- comma-separated-values file
      SKIP_HEADER     = 1      -- skip the single header row
      FIELD_DELIMITER = ','
      TRIM_SPACE      = TRUE
      NULL_IF         = ('', 'NULL')
      EMPTY_FIELD_AS_NULL = TRUE;

- **TYPE = CSV** — tells Snowflake to parse the file as delimited text.
- **SKIP_HEADER = 1** — ignores row 1 (the column names) so it isn't loaded as a data row.
- **FIELD_DELIMITER = ','** — fields are separated by commas.
- **TRIM_SPACE / NULL_IF / EMPTY_FIELD_AS_NULL** — strip stray whitespace and turn blanks or the literal text `NULL` into real NULLs.
</details>

### Now create the table to load into

The FILE FORMAT describes *how to read* the file. Next you need somewhere to put the data: a table whose columns line up with the file's fields. `COPY INTO` maps them **by position** — field 1 → column 1, field 2 → column 2, and so on — so the column order below must match the order you saw in the peek (`customer_id`, `signup_date`, `channel`), and each column needs a type that fits its values.

Fill in the type for columns 1 and 3. `signup_date` is set to `DATE` for you so you can do date math on it later.

In [ ]:
-- Fill in the column types based on what the peek showed you
CREATE OR REPLACE TABLE {{my_db}}.RAW.LOYALTY_SIGNUPS (
  customer_id  ___,         -- column 1: numeric IDs — fill in the type
  signup_date  DATE,        -- column 2: dates must be the DATE type (not text) so you can do date math later — filled in for you
  channel      ___          -- column 3: short text values — fill in the type
);


> <details><summary>📖 Reference solution</summary>
>
>     customer_id  INT,      -- numeric IDs
>     signup_date  DATE,     -- (given) dates, so you can do date math
>     channel      VARCHAR   -- short text values ('app', 'web', 'in-store')
> </details>

### Now load it — with CoCo's help

You've got a stage, a file format, and an empty table. The last step is the `COPY INTO` that moves the file into the table. Rather than us handing you the syntax, **ask CoCo to write it** — this is exactly how you'll work day to day.

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">Write me a Snowflake COPY INTO statement that loads the file loyalty_signups.csv from the stage @PLATPLUS_DB.ROASTERY_RAW.LOAD_STAGE into my table {{my_db}}.RAW.LOYALTY_SIGNUPS, using the file format {{my_db}}.RAW.CSV_FMT. Return just the statement.</pre>

Paste CoCo's answer into the empty cell below and run it. Reference solution is just below the cell.

In [ ]:
-- Paste CoCo's COPY INTO statement on the blank line below, then run this cell


<details>
<summary>📖 Reference solution</summary>

    COPY INTO {{my_db}}.RAW.LOYALTY_SIGNUPS
    FROM @PLATPLUS_DB.ROASTERY_RAW.LOAD_STAGE/loyalty_signups.csv
    FILE_FORMAT = (FORMAT_NAME = {{my_db}}.RAW.CSV_FMT);

</details>

In [ ]:
SELECT COUNT(*) AS rows_loaded FROM {{my_db}}.RAW.LOYALTY_SIGNUPS;


> 🖥 **Snowsight alternative:** You can load CSVs without writing any SQL. In Snowsight, open your database → RAW schema → click **Load Data**. The wizard detects delimiter and header row automatically and generates the FILE FORMAT and COPY INTO for you. Either path produces the same table — the SQL route above gives you visibility into exactly what the wizard is doing.

> :mag_right: **Review the output.** The COPY INTO result shows rows loaded, parse errors (should be 0), and file status (LOADED). The COUNT(*) confirms how many rows made it in.

> :point_right: **Your turn.** Run a `SELECT * FROM {{my_db}}.RAW.LOYALTY_SIGNUPS LIMIT 10` below — check that `customer_id` is numeric, `signup_date` is a date, and `channel` contains the three expected values.

In [ ]:
-- Your turn: preview the loaded data
SELECT * 
FROM {{my_db}}.RAW.LOYALTY_SIGNUPS 
LIMIT 10;


---
## 3 · Connect to the Supplier Stage

The loyalty sign-ups lived in an internal stage — data your team uploaded directly into Snowflake. But many real-world data sources come from *outside*: a vendor, a partner, or a cloud-based system that writes files to its own cloud storage bucket.

**The scenario:** Roastery's coffee bean suppliers send quarterly product catalog updates to a dedicated stage. The latest Q4 2025 catalog just arrived. Let's connect to it and load it.

Start by listing the stage to see what files the supplier has dropped.

In [ ]:
LIST @PLATPLUS_DB.ROASTERY_RAW.SUPPLIER_STAGE;


> :mag_right: **Review the output.** You'll see four files: `roastery_products_q4_2025.csv` (the clean catalog — focus here), `roastery_products_q4_2025_errors.csv` (bad rows, used in Section 4), `roastery_products_q4_2025_quoted.csv` (a quoted-field variant used in the Platform Admin lab — ignore it here), and `roastery_loyalty_events.json` (used in the Data Engineer lab).

In [ ]:
-- Raw preview of the supplier catalog
SELECT $1, $2, $3, $4, $5, $6
FROM @PLATPLUS_DB.ROASTERY_RAW.SUPPLIER_STAGE/roastery_products_q4_2025.csv
LIMIT 4;


> :mag_right: **Review the output.** Row 1 is the header: `product_id`, `product_name`, `category`, `origin_country`, `cost_per_unit`, `active`. The values look clean — numeric product IDs, text categories, decimal costs, TRUE/FALSE active flags.

### Reuse the format you already have

This file is another comma-delimited CSV with a header row — structurally the same as the loyalty file. So you **don't need to define a new file format**: the `CSV_FMT` you created in Section 2 works here too. *Define once, reuse everywhere* is the whole point of a FILE FORMAT object.

Create the target table below, then load it.

In [ ]:
CREATE OR REPLACE TABLE {{my_db}}.RAW.SUPPLIER_PRODUCTS (
  product_id      INT,
  product_name    VARCHAR(100),
  category        VARCHAR(50),
  origin_country  VARCHAR(50),
  cost_per_unit   FLOAT,
  active          BOOLEAN
);


> :point_right: **Your turn — you write this one.** You watched CoCo write the loyalty `COPY INTO` in Section 2. This is the same statement with a different file and table. Write it yourself in the cell below: load `roastery_products_q4_2025.csv` from `@PLATPLUS_DB.ROASTERY_RAW.SUPPLIER_STAGE` into `SUPPLIER_PRODUCTS`, **reusing `CSV_FMT`**.

Stuck or want to check your work? The reference solution is just below the cell.

In [ ]:
-- Your turn: write the COPY INTO for the supplier catalog (reuse CSV_FMT)

<details>
<summary>📖 Reference solution</summary>

    COPY INTO {{my_db}}.RAW.SUPPLIER_PRODUCTS
    FROM @PLATPLUS_DB.ROASTERY_RAW.SUPPLIER_STAGE/roastery_products_q4_2025.csv
    FILE_FORMAT = (FORMAT_NAME = {{my_db}}.RAW.CSV_FMT);

</details>

> :warning: **Getting "Numeric value 'product_id' is not recognized"?** That means the header row is being loaded as data — your `CSV_FMT` is missing `SKIP_HEADER = 1`. Fix the format above, then re-run the `COPY INTO` cell. Paste the error into CoCo if you want it to walk you through the fix.

In [ ]:
-- Spot-check the data itself (the COPY INTO output above already reported the row count)
SELECT * FROM {{my_db}}.RAW.SUPPLIER_PRODUCTS LIMIT 5;


> :mag_right: **Review the output.** The COPY INTO result above already confirmed the load — check its **`rows_parsed`** and **`rows_loaded`** (50 each, 0 errors). This cell is a quick spot-check of the actual data: categories include `Single Origin Beans`, `Blends`, `Cold Brew`, `Espresso`, `Equipment`, and `Merchandise`.

---
## 4 · What Happens When a File Has Bad Rows

The supplier also sent the same catalog with two rows that have formatting problems — `cost_per_unit` contains `N/A` and `PENDING` instead of numbers. Sooner or later a file you're handed will have junk in it, so let's see what Snowflake does about it.

Try loading that file into your `SUPPLIER_PRODUCTS` table (where `cost_per_unit` is a FLOAT) and watch what happens.

**Why this matters for analysts:** By default Snowflake rejects the *entire* load if any row is bad (`ON_ERROR = ABORT_STATEMENT`). That's usually what you want — a clean all-or-nothing load beats a half-filled table you can't trust. The skill is reading the error and knowing whose problem it is.

In [ ]:
-- This load will FAIL on purpose — read the error message it returns
COPY INTO {{my_db}}.RAW.SUPPLIER_PRODUCTS
FROM @PLATPLUS_DB.ROASTERY_RAW.SUPPLIER_STAGE/roastery_products_q4_2025_errors.csv
FILE_FORMAT = (FORMAT_NAME = {{my_db}}.RAW.CSV_FMT);


> :mag_right: **Read the error.** The load aborts and Snowflake tells you why — `COST_PER_UNIT` received a text value (`N/A` or `PENDING`) where a number was expected, and it names the row. Your `SUPPLIER_PRODUCTS` table is untouched: nothing was loaded, so nothing is half-broken.

> :bulb: **Whose problem is it?** This is a *source data* problem, not a load problem — the file is wrong, your SQL is fine. The real-world fix is to send the file back to the supplier to correct those two rows. Resist the urge to force it through with `ON_ERROR = CONTINUE` — a partial load you can't fully account for is harder to trust than a clean rejection.

> 📊 **Your angle starts here.** Everything above was shared foundation — staging, formats, COPY INTO. From here on, you apply it as an analyst: turning loaded data into answers, charts, and shareable views.

---
## 5 · Your angle — Data Analyst: Who's signing up for what?

You now have two tables in your schema: `LOYALTY_SIGNUPS` (which channels brought customers in) and `SUPPLIER_PRODUCTS` (what the Roastery sells). Neither is useful on its own. Together, they start telling a story.

First, create a reusable view over the sign-up data — you'll use it in downstream queries.

In [ ]:
CREATE OR REPLACE VIEW {{my_db}}.RAW.SIGNUPS_BY_CHANNEL AS
SELECT
    channel,
    COUNT(*)                                          AS signups,
    MIN(signup_date)                                  AS first_signup,
    MAX(signup_date)                                  AS latest_signup,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_of_total
FROM {{my_db}}.RAW.LOYALTY_SIGNUPS
GROUP BY channel
ORDER BY signups DESC;


**Capturing a query ID.** Every query Snowflake runs gets a **unique query ID** — a session-level identifier you can harvest and reuse programmatically. `LAST_QUERY_ID()` returns the ID of the most recent query, and `SET query_id = ...` stores it as a **session variable** you can reference with `$query_id` in any later cell of this session. We'll use it just below to replay these exact results without re-running the query.

In [ ]:
SELECT * FROM {{my_db}}.RAW.SIGNUPS_BY_CHANNEL;
SET query_id = LAST_QUERY_ID();
SELECT * FROM TABLE(RESULT_SCAN($query_id));


> :mag_right: **Review the output.** Which channel drives the most sign-ups? Which has the earliest first_signup date — meaning it was the first channel you acquired customers through?

In [ ]:
-- What does the supplier catalog look like by category?
-- Average cost, product count, and active vs inactive breakdown
SELECT
    category,
    COUNT(*)                                          AS total_products,
    SUM(CASE WHEN active THEN 1 ELSE 0 END)           AS active_products,
    ROUND(AVG(cost_per_unit), 2)                      AS avg_cost,
    ROUND(MIN(cost_per_unit), 2)                      AS min_cost,
    ROUND(MAX(cost_per_unit), 2)                      AS max_cost
FROM {{my_db}}.RAW.SUPPLIER_PRODUCTS
GROUP BY category
ORDER BY avg_cost DESC;


> :mag_right: **Review the output.** Which category has the highest average cost? Which has the most products? Are there any inactive products — and in which category?

> :point_right: **Your turn.** Modify the query to show only **active** products, sorted by `avg_cost` descending. Which active category would you prioritise for a margin analysis?

---
## Chart it, then share it

A table of numbers is fine for you — but stakeholders usually want a picture. You don't need any code for this: **every SQL cell result has a built-in Chart view.**

**Replaying results with RESULT_SCAN.** In the cell above you captured a `query_id` — a unique identifier Snowflake assigns to every query. `RESULT_SCAN($query_id)` brings that result set back as a table you can re-query, with no recompute needed.

> :bulb: **Result cache:** By default, if you re-issue the *exact same* query within **24 hours** and the underlying data hasn't changed, Snowflake returns the previously computed results from cache — rather than re-running the query — provided you have the required privileges. `RESULT_SCAN` is a direct way to tap into this.
>
> Docs: [RESULT_SCAN](https://docs.snowflake.com/en/sql-reference/functions/result_scan) · [LAST_QUERY_ID](https://docs.snowflake.com/en/sql-reference/functions/last_query_id) · [Using persisted query results](https://docs.snowflake.com/en/user-guide/querying-persisted-results)

> :point_right: **Your turn — build the chart.** Run the SQL cell below to replay your sign-up summary. In its result, switch from the **Results** tab to the **Chart** tab, then set:
> - **Chart type:** Bar
> - **X-axis:** `CHANNEL`
> - **Y-axis:** `SIGNUPS`
> - **Aggregate:** NONE
> - **Group by:** NONE

In [ ]:
SELECT * FROM TABLE(RESULT_SCAN($query_id));


That gives you a bar chart of sign-ups by channel — the tallest bar is your top acquisition channel. A **bar chart** is the right pick here because you're comparing a count across a handful of distinct categories. Try switching the Y-axis to `PCT_OF_TOTAL` to see the same story as a share of the whole.

### Export results to send to others

Charts live in the notebook — but sometimes a colleague just wants the numbers in a spreadsheet. Any query result can be exported straight from the results grid, no extra SQL required.

> 🖥 **Download from Snowsight:** Return to the `replay_signups_by_channel` cell above and select **Table** output. In the results panel, click the **download** (⬇) button and choose **Download as .csv**. You'll get a file you can attach to an email or drop into a slide deck.

---
### A more complex view: signups over time

A **bar chart** is great for comparing categories side by side. A **line chart** is the right pick when you want to show *change over time* — peaks, dips, and trends.

To build one you first need to **bucket** your dates. `DATE_TRUNC('month', signup_date)` rounds each date down to the first of its month, so you can `GROUP BY` month and count sign-ups per period. This is one of the most common patterns in analytics.

In [ ]:
SELECT DATE_TRUNC('month', signup_date) AS signup_month,
       COUNT(*) AS signups
FROM {{my_db}}.RAW.LOYALTY_SIGNUPS
GROUP BY 1
ORDER BY 1;


> :point_right: **Your turn — build the line chart.** In the result above, open the **Chart** tab and set:
> - **Chart type:** Line
> - **X-axis:** `SIGNUP_MONTH`
> - **Y-axis:** `SIGNUPS`
> - **Aggregate:** NONE
>
> This shows how sign-up volume has changed month by month. Look for the steepest rise — that's when acquisition was strongest. Compare it to the bar chart: *same data, different question.*

---
## Verify Your Work

In [ ]:
checks = [
    ('LOYALTY_SIGNUPS loaded', f'SELECT COUNT(*) FROM {my_db}.RAW.LOYALTY_SIGNUPS', 1),
    ('SUPPLIER_PRODUCTS loaded', f'SELECT COUNT(*) FROM {my_db}.RAW.SUPPLIER_PRODUCTS', 1),
    ('SIGNUPS_BY_CHANNEL view created', f"SELECT COUNT(*) FROM INFORMATION_SCHEMA.VIEWS WHERE TABLE_SCHEMA='RAW' AND TABLE_NAME='SIGNUPS_BY_CHANNEL' AND TABLE_CATALOG='{my_db}'", 1),
]
all_passed = True
for label, sql, min_val in checks:
    try:
        result = session.sql(sql).collect()[0][0]
        ok = result >= min_val
        print(('✅' if ok else '⛔') + f' {label}: {result}')
        if not ok:
            all_passed = False
    except Exception as e:
        print(f'⛔ {label}: ERROR — {e}')
        all_passed = False
if all_passed:
    print('\n🎉 All checks passed — Section 4\'s bad-rows load aborted, so your product table stayed clean.')


---
### Check your understanding 🧠
A few questions on this lab's key themes. Run each cell, choose an answer, and review the feedback.

In [ ]:
from resources.knowledgecheck import data_analyst_quiz as quiz
quiz.display_question("q1")

In [ ]:
from resources.knowledgecheck import data_analyst_quiz as quiz
quiz.display_question("q2")

In [ ]:
from resources.knowledgecheck import data_analyst_quiz as quiz
quiz.display_question("q3")

In [ ]:
from resources.knowledgecheck import data_analyst_quiz as quiz
quiz.display_question("q4")

In [ ]:
from resources.knowledgecheck import data_analyst_quiz as quiz
quiz.display_question("q5")

---
### :snowflake: Key Takeaways

- `LIST @stage` and `SELECT $1` from a staged file let you **inspect before you load** — essential for debugging
- A **FILE FORMAT** object defines delimiter, header handling, null treatment — **define once, reuse** across many COPY INTO loads
- **COPY INTO** loads a staged file into a typed table; the FILE FORMAT must match the file's actual structure
- CoCo can **write your COPY INTO** for you — and once you've seen the shape, you can write the next one unaided
- A bad row makes the **whole load abort by default** (`ON_ERROR = ABORT_STATEMENT`) — read the error, and if it's source data, send the file back
- Every SQL cell result has a built-in **Chart** view (bar, line, etc.) and can be **exported to CSV** — no code needed to visualise or share a result

**The Other Personas:**
- The **Data Engineer** notebook covers ON_ERROR handling, JSON FLATTEN, and scheduling COPY INTO with a task
- The **Platform Admin** notebook covers SHOW STAGES, COPY_HISTORY, and the storage integration admin walkthrough

> :thinking: **Transfer it.** Next week a partner sends you a **pipe-delimited** (`|`) file with no header. Which *one* part of your FILE FORMAT changes — and what do you set `SKIP_HEADER` to?
>
> <details><summary>Answer</summary>
>
> Change `FIELD_DELIMITER='|'` and set `SKIP_HEADER=0` (no header row). Everything else stays the same — that's the point of a reusable FILE FORMAT.
>
> </details>

### Notebook completion. 🏁
You're done — your notebook is saved automatically.

> 💡 **Moving on to another lab?** Shut down this notebook's **kernel** — click **Connected** at the top → **Shut down kernel**. All your lab notebooks share one compute service, so clearing a finished kernel frees up memory for the next lab. Your service stays running, ready in seconds.

> 💡 **Finished for the day?** **Suspend your service** — click **Connected** → hover your service name → **Suspend**. This stops compute; it resumes automatically next time you run a cell.